# DD1328 Betygshöjande extrauppgift

I denna uppgift kommer du att skapa ett **Bloomfilter**.

Godkänd uppgift höjer slutbetyget i kursen med ett **betygssteg**.

Uppgiften redovisas genom att ladda upp en ifylld version av denna notebook-fil till ditt github-repo senast **2026-06-05**. Om ett ärligt försök gjorts men vissa detaljer saknas eller är felaktiga så kan komplettering utföras inom en vecka från att bedömning skett. 

Ingen muntlig redovisning behövs för denna uppgift.

---

### Bloomfilter

Ett bloomfilter är en *probabilistisk datastruktur*, för att implementera en *mängd* / ett *set*, där vi kan lägga in element och sedan avgöra definitivt om ett visst element **inte** har lagts in, och med hög sannolikhet avgöra om ett element **har** lagts in. Och detta görs utan att faktiskt lagra själva elementen i datastrukturen, vilket spar på mängden minne vi behöver använda. Läs på om tanken bakom bloomfilter här: https://yourbasic.org/algorithms/bloom-filter/

Ert bloomfilter ska ha de publika metoderna:

* en konstruktor **\_\_init\_\_(self, m, k)** som skapar upp ett bloomfilter givet en storlek *m* (int) och antalet hashfunktioner *k* (int)
* **add(s)** - För att lägga till ett element *s* (sträng)
* **contains()** - För att kolla om elementet "finns" i bloomfiltret (alltså, om vi någonsin anropat add med elementet) eller inte

Vi behöver som tur är inte faktiskt skapa k stycken olika hashfunktioner själva (och vi vill ju att man ska kunna skapa bloomfilter med olika värden på k). Därför kommer ni istället att utnyttja något som kallas för *double hashing*, ett sätt för att undvika krockar i en vanlig hashtabell som vi även kan använda här för att simulera flera hashfunktioner. 

Du behöver endast implementera **två** olika hashfunktioner som du väljer själv. De behöver inte vara något cutting edge, men de ska ge bättre spridning än den hasfunktion vi använde i LABC. Det viktiga är att de är två olika. Gör lite efterforskningar kring hashfunktioner för strängar och välj ut två, här är några exempel på hashfunktioner som ni kan kolla upp:

* Weighted sum hash
* Polynomial rolling hash
* Folding hash
* djb2
* sdbm

Det går också bra att kombinera dessa till någon egen variant om ni så vill.

Implementera dina funktioner och lägg in i blocket nedan:

In [ ]:
# Implementation av valda hashfunktioner

# Each left bit shift means to multiply by *2, so
# (h << 5) + h = 33 * h
# (h << 6) + (h << 16) - h = 65599 * h
# ord(c) returns pretty much the ascii value of c

def djb2(s):
    h = 5381

    for c in s:
        # h = ord(c) + ((h << 5) + h) # ascii + 33*h
        h = ord(c) + 33*h

    return h

def sdbm(s):
    h = 0

    for c in s:
        # h = ord(c) + (h << 6) + (h << 16) - h # ascii + 65599*h
        h = ord(c) + 65599*h

    return h

**FRÅGA:** Hur fungerar dina valda hasfunktioner? Hur ser de till att det blir bra spridning över tabellen? Ge en utförlig beskrivning av båda (fyll i blocket nedan)

När du nu har de två hashfunktionerna ($h_1(x)$ och $h_2(x)$), och (snart) har en bit-array i bloomfiltret med storlek $m$ så kan du simulera ett godtyckligt antal hashfunktioner $h_i(x)$ på följande vis:

$$ h_i(x)=(h_1(x)+i⋅h_2(x))\: mod \: m $$

**FRÅGA:** Vilket problem uppstår i de fall $h_2(x) = 0$ ? (fyll i blocket nedan)

In [ ]:
# Då blir hashfunktionen h_i(x) samma för alla i, nämligen h_i(x) = h_1(x) mod m
# Därför skulle alla "olika" hashfunktioner mappa till samma index i bloomfiltret (samma som om vi bara använde en enda hashfunktion)

Implementera nu själva bloomfiltret genom att fylla i följande kodskelett och även lägga in dina hashfunktioner och deras sammanslagning (så att vi kan få fram $h_i$). Skriv även ett huvudprogram som demonstrerar att det fungerar.

In [ ]:
class BloomFilter:
    def __init__(self, size_m, num_hash_functions_k):
        self.bits = [False] * size_m
        self.k = num_hash_functions_k
        
    def add(self, string):
        for idx in self._mapped(string):
            self.bits[idx] = True
        
    def contains(self, string):
        for idx in self._mapped(string):
            if self.bits[idx] == False:
                return False # Definitely does NOT exist
        return True # Probably exists
    
    def _mapped(self, x):
        for i in range(self.k):
            yield self._hash(x, i)
    
    def _hash(self, x, i):
        return (self._djb2(x) + i * self._sdbm(x)) % len(self.bits)

    @staticmethod
    def _djb2(s):
        h = 5381

        for c in s:
            # h = ord(c) + ((h << 5) + h) # ascii + 33*h
            h = ord(c) + 33*h

        return h

    @staticmethod
    def _sdbm(s):
        h = 0

        for c in s:
            # h = ord(c) + (h << 6) + (h << 16) - h # ascii + 65599*h
            h = ord(c) + 65599*h

        return h

bloomFilter = BloomFilter(10, 3) # Small to also detect some false positives
print("Actions: 'a' add(), 'c' contains(), 'q' quit")
print("Input two strings every time for add() and contains(): 'action string'")
while True:
    parts = input().split()
    try:
        command = parts[0]

        if command == 'a':
            bloomFilter.add(parts[1])

        elif command == 'c':
            if bloomFilter.contains(parts[1]):
                print("probably")
            else:
                print("no")

        elif command == 'q':
            break
    except:
        print("Invalid input. try again")

Actions: 'a' add(), 'c' contains(), 'q' quit
Input two strings every time for add() and contains(): 'action string'
probably
probably


**FRÅGOR:** (fyll i blocket nedan) 
* Vad blir den asymptotiska tidskomplexiteten för add() och contains() med avseende på antal hashfunktioner k? Hur väl stämmer den överens med den verkliga körtiden givet att vi simulerar flera olika k enligt instruktionen ovan?
* Vi skulle ju kunna implementera en mängd (alltså kunna lägga in element och sedan kontrollera om de är en del av mängden eller inte) med en vanlig hashtabell också. Diskutera för- och nackdelar med dessa två datastrukturer i det här sammanhanget.

In [ ]:
# Fråga 1:

# hash(x, i) tar O(1) tid eftersom vi alltid kallar _djb2(s) och _sdbm(s) exakt en gång var.
# _mapped(x) tar O(k) tid eftersom den kallar _hash(x, i) k gånger.
# Alltså tar både add(string) och contains(string) O(k) tid i värsta fall.
# add() tar O(k) tid oberoende av bästa/värsta fall.
# contains() tar i bästa fall O(1) tid ifall en early exit sker vid "self.bits[idx] == False" för i = 1.

# TODO: Svara på "Hur väl stämmer den överens med den verkliga körtiden givet att vi simulerar flera olika k enligt instruktionen ovan?"

# Fråga 2:

# En hashtabell är EJ probabilistisk, dvs ger ALLTID definitiva JA/NEJ-svar.
# Däremot tar en hashtabell MARKANT mer minne, och bör därför endast väljas framför ett bloom-filter ifall:
# 1. den probabilistiska avvägningen ej är acceptabel, ELLER 2. tabellen kan behöva utökas framöver, ELLER 3. element kan behöva tas bort framöver.
# (det är i alla fall de tre undantag jag kommer att tänka på)

### Borttagning av element

En nackdel med Bloomfilter av den vanliga varianten är att vi inte kan ta bort element! **Förklara varför i blocket nedan**(om du inte redan gjort det i förra frågan):

Eftersom alla tillagda element delar bits med varandra skulle borttagning av ett element innebära att alla andra tillagda element som
råkar dela samma mappade bits inte längre räknas tillhöra bloom-filtret.

Exempel:
add("Hej") sätter boolean nr2 till True.
remove("Hejdå") sätter boolean nr2 till False.
contains("Hej") returnerar nu FELAKTIGT False eftersom boolean nr2 == False

Nu ska vi åtgärda detta tillkortakommande genom att utöka vårt bloomfilter till ett *Counting Bloom Filter*. Här har vi istället för 1/0 (True/False) på varje position i tabellen istället en räknare (ett heltal) som vi kan inkrementera och dekrementera. Vi använder alltså mer minne för att utöka funktionaliteten. När vi lägger in ett element så ökar vi räknaren med ett på alla positioner som hashfunktionerna pekar på, och när vi tar bort minskar vi alla med ett. För att kolla om ett element finns så kontrollerar vi att alla positioner har ett värde $> 0$. Gör dessa ändringar i din tabell från tidigare i uppgiften och lägg till en metod **delete(self, string)** som tar bort element. Skriv även ett huvudprogram som demonstrerar att det fungerar.

In [ ]:
class CountingBloomFilter:
    def __init__(self, size_m, num_hash_functions_k):
        self.integers = [0] * size_m
        self.k = num_hash_functions_k
        
    def add(self, string):
        for idx in self._mapped(string):
            self.integers[idx] += 1

    def delete(self, string):
        for idx in self._mapped(string):
            self.integers[idx] -= 1
            
    def contains(self, string):
        for idx in self._mapped(string):
            if self.integers[idx] < 1:
                return False # Definitely does NOT exist
        return True # Probably exists
    
    def _mapped(self, x):
        for i in range(self.k):
            yield self._hash(x, i)
    
    def _hash(self, x, i):
        return (self._djb2(x) + i * self._sdbm(x)) % len(self.integers)

    @staticmethod
    def _djb2(s):
        h = 5381

        for c in s:
            # h = ord(c) + ((h << 5) + h) # ascii + 33*h
            h = ord(c) + 33*h

        return h

    @staticmethod
    def _sdbm(s):
        h = 0

        for c in s:
            # h = ord(c) + (h << 6) + (h << 16) - h # ascii + 65599*h
            h = ord(c) + 65599*h

        return h

bloomFilter = CountingBloomFilter(10, 3) # Small to also detect some false positives
print("Actions: 'a' add(), 'c' contains(), 'd' delete() 'q' quit")
print("Input two strings every time for add(), contains(), and delete(): 'action string'")
while True:
    parts = input().split()
    try:
        command = parts[0]

        if command == 'a':
            bloomFilter.add(parts[1])

        elif command == 'c':
            if bloomFilter.contains(parts[1]):
                print("probably")
            else:
                print("no")

        elif command == 'd':
            bloomFilter.delete(parts[1])
            
        elif command == 'q':
            break
    except:
        print("Invalid input. try again")

**SISTA FRÅGAN:** Vad händer om vi försöker ta bort ett element som aldrig lagts in? (fyll i blocket nedan)

Isåfall kan ett snarlikt problam som borttagningsproblemet med bit-implementationen av bloom-filtret uppstå.

Exempel:
add("Hej") inkrementerar integer nr2 till 1.
remove("Hejdå") dekrementerar integer nr2 till 0.
contains("Hej") returnerar nu FELAKTIGT False eftersom integer nr2 < 1